In [2]:
from collections import Counter
import re

# -------------------------
# Read Corpus File
# -------------------------
with open("corpus.txt", "r") as file:
    corpus = file.read()

print("===== Sample Corpus =====")
print(corpus)

# -------------------------
# Create Initial Vocabulary
# -------------------------
words = corpus.split()

vocab = Counter()

for word in words:
    chars = " ".join(list(word)) + " </w>"
    vocab[chars] += 1

print("\n===== Initial Vocabulary =====")

for word, freq in vocab.items():
    print(f"{word} : {freq}")

# -------------------------
# Find Pair Frequencies
# -------------------------
def get_stats(vocab):

    pairs = Counter()

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq

    return pairs

# -------------------------
# Merge Best Pair
# -------------------------
def merge_vocab(pair, vocab):

    new_vocab = Counter()

    pattern = re.escape(" ".join(pair))
    replacement = "".join(pair)

    for word in vocab:

        new_word = re.sub(pattern, replacement, word)

        new_vocab[new_word] = vocab[word]

    return new_vocab

# -------------------------
# Train Tokenizer
# -------------------------
num_merges = 10

merge_rules = []

print("\n===== Training BPE Tokenizer =====")

for i in range(num_merges):

    pairs = get_stats(vocab)

    if len(pairs) == 0:
        break

    best_pair = max(pairs, key=pairs.get)

    print(f"\nMerge {i+1}")
    print("Best Pair :", best_pair)

    vocab = merge_vocab(best_pair, vocab)

    merge_rules.append(best_pair)

# -------------------------
# Final Vocabulary
# -------------------------
print("\n===== Final Vocabulary =====")

for word in vocab:
    print(word)

print("\nVocabulary Size :", len(vocab))

# -------------------------
# Encode Function
# -------------------------
def encode(word):

    tokens = list(word) + ["</w>"]

    for pair in merge_rules:

        i = 0

        while i < len(tokens)-1:

            if tokens[i] == pair[0] and tokens[i+1] == pair[1]:

                tokens[i:i+2] = ["".join(pair)]

            else:

                i += 1

    return tokens

# -------------------------
# Decode Function
# -------------------------
def decode(tokens):

    text = "".join(tokens)

    text = text.replace("</w>", "")

    return text

# -------------------------
# Testing
# -------------------------
print("\n===== Testing =====")

test_words = ["lowest", "newer", "widest"]

for word in test_words:

    encoded = encode(word)

    decoded = decode(encoded)

    print("\nOriginal :", word)
    print("Encoded  :", encoded)
    print("Decoded  :", decoded)

# -------------------------
# Merge Rules
# -------------------------
print("\n===== Learned Merge Rules =====")

for i, rule in enumerate(merge_rules, start=1):

    print(f"{i}. {rule}")

print("\nProject Completed Successfully!")

===== Sample Corpus =====
low
lower
lowest
new
newer
newest
wide
wider
widest
low
lower
new
wide

===== Initial Vocabulary =====
l o w </w> : 2
l o w e r </w> : 2
l o w e s t </w> : 1
n e w </w> : 2
n e w e r </w> : 1
n e w e s t </w> : 1
w i d e </w> : 2
w i d e r </w> : 1
w i d e s t </w> : 1

===== Training BPE Tokenizer =====

Merge 1
Best Pair : ('l', 'o')

Merge 2
Best Pair : ('lo', 'w')

Merge 3
Best Pair : ('e', 'r')

Merge 4
Best Pair : ('er', '</w>')

Merge 5
Best Pair : ('n', 'e')

Merge 6
Best Pair : ('ne', 'w')

Merge 7
Best Pair : ('w', 'i')

Merge 8
Best Pair : ('wi', 'd')

Merge 9
Best Pair : ('e', 's')

Merge 10
Best Pair : ('es', 't')

===== Final Vocabulary =====
low </w>
low er</w>
low est </w>
new </w>
new er</w>
new est </w>
wid e </w>
wid er</w>
wid est </w>

Vocabulary Size : 9

===== Testing =====

Original : lowest
Encoded  : ['low', 'est', '</w>']
Decoded  : lowest

Original : newer
Encoded  : ['new', 'er</w>']
Decoded  : newer

Original : widest
Encoded  : [